In [6]:
import os, sys
sys.path.append(os.path.abspath(".."))  # adjust so `src` is importable

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from src import config
from src.data_pipeline import (
    load_merged_dataframe, get_column_groups, split_partitions,
    normalize_targets, make_dataset,
)
from src.model import build_model, compile_model
from src.training import train_in_sessions

print("TF version:", tf.__version__)

TF version: 2.20.0


In [7]:
df = load_merged_dataframe()
attr_cols, bbox_cols, landmark_cols = get_column_groups(df)
train_df, val_df, test_df = split_partitions(df)

train_df = normalize_targets(train_df, bbox_cols, landmark_cols)
val_df   = normalize_targets(val_df, bbox_cols, landmark_cols)
test_df  = normalize_targets(test_df, bbox_cols, landmark_cols)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"attrs={len(attr_cols)}  landmarks={len(landmark_cols)}  bbox={len(bbox_cols)}")

train=162770  val=19867  test=19962
attrs=40  landmarks=10  bbox=4


In [8]:
train_ds = make_dataset(train_df, attr_cols, bbox_cols, landmark_cols, shuffle=True)
val_ds   = make_dataset(val_df,   attr_cols, bbox_cols, landmark_cols)

In [9]:
model, backbone = build_model(
    img_size=config.IMG_SIZE,
    num_attrs=len(attr_cols),
    num_landmarks=len(landmark_cols),
    num_bbox=len(bbox_cols),
)
model = compile_model(model)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_1… │ (None, 4, 4,      │  2,257,984 │ input_layer[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    327,936 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attributes (Dense)  │ (None, 40)        │     10,280 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ landmarks (Dense)   │ (None, 10)        │      2,570 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox (Dense)        │ (None, 4)         │      1,028 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,599,798 (9.92 MB)

 Trainable params: 341,814 (1.30 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
model = train_in_sessions(
    model, train_ds, val_ds,
    callbacks=[],       # no cross-session callbacks — see explanation above
    stage_name="normal",
)

[normal] Starting fresh training (target: 15 epochs)
Epoch 1/2
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 2279s 887ms/step - attributes_accuracy: 0.0134 - attributes_loss: 0.3388 - bbox_loss: 0.5876 - bbox_mae: 0.3857 - landmarks_loss: 0.0017 - landmarks_mae: 0.0228 - loss: 3.2861 - val_attributes_accuracy: 0.0119 - val_attributes_loss: 0.3044 - val_bbox_loss: 0.5994 - val_bbox_mae: 0.3870 - val_landmarks_loss: 2.5437e-04 - val_landmarks_mae: 0.0101 - val_loss: 3.3035
Epoch 2/2
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 2493s 975ms/step - attributes_accuracy: 0.0149 - attributes_loss: 0.3164 - bbox_loss: 0.5812 - bbox_mae: 0.3774 - landmarks_loss: 2.7334e-04 - landmarks_mae: 0.0104 - loss: 3.2239 - val_attributes_accuracy: 0.0143 - val_attributes_loss: 0.2952 - val_bbox_loss: 0.5965 - val_bbox_mae: 0.3760 - val_landmarks_loss: 2.3905e-04 - val_landmarks_mae: 0.0095 - val_loss: 3.2795
[normal] Completed epochs 1-2/15 — train loss: 3.2239, val loss: 3.2795


In [11]:
import json
with open(os.path.join(config.CHECKPOINT_DIR, "state.json")) as f:
    print(json.load(f))

{'completed_epochs': 2, 'done': False}
